# Burst 2 (~45 CUs on NVIDIA L4 24GB): Week 2 / Prototype 1 — 1,000-Prompt Residual Stream Sweep, Linear Hallucination Probe, `Outlines` Guided JSON & Runtime Steering

**Sprint**: Applied AI Foundry (v4.0) — **Prototype 1 of the 3-Prototype Startup Tournament**  
**Target HF Dataset**: [`imhlg/finops-residual-activations-1k`](https://huggingface.co/datasets/imhlg/finops-residual-activations-1k)  
**Target HF Model/Probe Repo**: [`imhlg/qwen3-4b-activation-probes-week2`](https://huggingface.co/imhlg/qwen3-4b-activation-probes-week2)

### Architectural Pipeline
1. **Synthesize 1,000 Contrastive FinOps Claims**: 500 physically accurate cloud/GPU claims vs. 500 hallucinated claims across real GPU VRAM math, KV-cache limits, `vLLM` flags, and Kubernetes OOM diagnostics.
2. **Multi-Layer Residual Hook Sweep**: Attach `register_forward_hook` across Layers `[6, 12, 18, 24, 30, 35]` of `Qwen/Qwen3-4B` (`d_model=2560`) with `padding_side="left"` batched `bfloat16` passes on CUDA.
3. **Train `10 KB` Linear Hallucination Probe + CAA Steering Vector ($
ec{v}_{	ext{truth}}$)**: Measure 5-fold CV Precision/Recall/AUC across depths and isolate Layer 18's semantic decision boundary (`>95%` precision).
4. **Syntactic + Semantic Lock**: Combine **`Outlines`** grammar-constrained JSON decoding with **in-flight Layer 18 residual steering** and verify **`<2.0 ms`** hook overhead vs. `~500 ms` external LLM guardrails.

> **Colab Runtime**: Select **Runtime → Change runtime type → L4 GPU**, verify `HF_TOKEN` is in Colab Secrets, and click **Runtime → Run all**. Disconnect immediately after Cell 7 pushes to Hugging Face!

In [ ]:
%pip install -q -U "transformers>=5.0.0" "datasets>=3.0.0" "huggingface_hub>=0.28.0" "accelerate>=1.2.0" "scikit-learn>=1.5.0" "safetensors>=0.4.5" "outlines>=0.1.0" "pydantic>=2.0.0"

import torch

assert torch.cuda.is_available(), (
    "CUDA GPU not detected! Go to Runtime -> Change runtime type and select L4 or A100 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16

print(f"GPU Detected       : {gpu_name} (Compute Capability SM {major}.{minor})")
print(f"Total GPU VRAM     : {vram_gb:.2f} GB")
print(f"Selected Precision : {compute_dtype}")

In [ ]:
import os
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Authenticated to Hugging Face Hub via Colab Secret HF_TOKEN.")
else:
    print("HF_TOKEN secret not found. Launching interactive notebook login...")
    notebook_login()

In [ ]:
# ==============================================================================
# PART 1: SYNTHESIZE 1,000 CONTRASTIVE FINOPS PROMPTS (500 TRUTHFUL / 500 HALLUCINATED)
# ==============================================================================
import json
from pathlib import Path
import random

MODEL_ID = "Qwen/Qwen3-4B"
DATASET_REPO_ID = "imhlg/finops-residual-activations-1k"
PROBE_REPO_ID = "imhlg/qwen3-4b-activation-probes-week2"
OUT_DIR = Path("outputs/week2_activation_probes")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def generate_1000_finops_claims(seed: int = 42) -> list[dict]:
    """Generate 500 grounded and 500 hallucinated infrastructure/FinOps statements."""
    rng = random.Random(seed)
    models = [
        ("Qwen3-4B", 4),
        ("Qwen2.5-7B", 7),
        ("Llama-3.1-8B", 8),
        ("Qwen2.5-14B", 14),
        ("Qwen2.5-32B", 32),
        ("Llama-3.1-70B", 70),
    ]
    gpus = [
        ("NVIDIA T4", 16),
        ("NVIDIA L4", 24),
        ("NVIDIA A10G", 24),
        ("NVIDIA A100-40GB", 40),
        ("NVIDIA A100-80GB", 80),
        ("NVIDIA H100-80GB", 80),
    ]
    contexts = [2048, 4096, 8192, 16384]
    concurrencies = [8, 16, 32, 64, 128]

    grounded_templates = [
        lambda m, b, g, v, ctx, c: (
            f"Audit Check: A {m} dense model in FP16 requires ~{b*2:.1f} GB just for static weights; "
            f"on a {v} GB {g}, quantized W4A16 (~{b*0.55:.1f} GB) leaves headroom for {c} concurrent {ctx}-token PagedAttention requests."
        ),
        lambda m, b, g, v, ctx, c: (
            f"Serving Note: Setting tokenizer.padding_side='left' with attention_mask ensures causal decoder batching "
            f"of {c} {m} prompts on {g} ({v} GB) samples from real final tokens rather than trailing <pad> IDs."
        ),
        lambda m, b, g, v, ctx, c: (
            f"FinOps Sizing: Serving {c} fine-tuned {m} variants via rank-8 LoRA adapters (~40 MB each) under vLLM --enable-lora "
            f"avoids duplicating {b*2:.1f} GB FP16 base checkpoints across separate {g} nodes."
        ),
        lambda m, b, g, v, ctx, c: (
            f"Latency Telemetry: Disabling latent reasoning via enable_thinking=False and using Outlines FSM regex logit masking "
            f"cuts {m} JSON extraction TTFT on {g} while enforcing 100% syntactic schema validity at {ctx} context."
        ),
        lambda m, b, g, v, ctx, c: (
            f"Memory Architecture: On {g} ({v} GB VRAM), GQA and FP8 KV-cache compression reduce KV memory per token, "
            f"preventing PodCrashLoopBackOff CUDA OOM at {c} concurrent {ctx}-token {m} sessions."
        ),
    ]

    hallucinated_templates = [
        lambda m, b, g, v, ctx, c: (
            f"Audit Check: A 70B FP16 dense model consumes only 6.2 GB VRAM and effortlessly sustains {c*4} concurrent "
            f"32,768-token streams inside a single 16 GB T4 GPU without quantization or offloading."
        ),
        lambda m, b, g, v, ctx, c: (
            f"Serving Note: Causal decoder-only transformers like {m} strictly require tokenizer.padding_side='right' "
            f"so the final index [-1] reads <pad> tokens to boost KV-cache throughput by 300% on {g}."
        ),
        lambda m, b, g, v, ctx, c: (
            f"FinOps Sizing: Attaching a rank-8 LoRA adapter to {m} rewrites all {b} billion base parameters in-place "
            f"and requires spinning up {c} isolated {g} GPUs because adapters cannot share base weights in vLLM."
        ),
        lambda m, b, g, v, ctx, c: (
            f"Latency Telemetry: PyTorch register_forward_hook executes an external REST call to GPT-4o on every token, "
            f"adding 750ms of mandatory network latency and permanently corrupting prior DynamicCache KV blocks."
        ),
        lambda m, b, g, v, ctx, c: (
            f"Memory Architecture: Outlines JSON decoding increases {m} VRAM requirements by 12x because FSM bitmasks "
            f"train a new 7B reward model on every {ctx}-token request on {g} ({v} GB)."
        ),
    ]

    rows = []
    # 500 Grounded (label=0) + 500 Hallucinated (label=1)
    for idx in range(500):
        m, b = models[idx % len(models)]
        g, v = gpus[(idx // len(models)) % len(gpus)]
        ctx = contexts[idx % len(contexts)]
        c = concurrencies[(idx // 3) % len(concurrencies)]
        t_fn = grounded_templates[idx % len(grounded_templates)]
        h_fn = hallucinated_templates[idx % len(hallucinated_templates)]

        rows.append({
            "id": f"truth_{idx:04d}",
            "text": t_fn(m, b, g, v, ctx, c),
            "label": 0,
            "label_name": "truthful",
            "model_ref": m,
            "gpu_ref": g,
            "split": "test" if idx >= 400 else "train",
        })
        rows.append({
            "id": f"halluc_{idx:04d}",
            "text": h_fn(m, b, g, v, ctx, c),
            "label": 1,
            "label_name": "hallucinated",
            "model_ref": m,
            "gpu_ref": g,
            "split": "test" if idx >= 400 else "train",
        })

    rng.shuffle(rows)
    return rows


records = generate_1000_finops_claims(seed=42)
n_train = sum(1 for r in records if r["split"] == "train")
n_test = sum(1 for r in records if r["split"] == "test")
print(f"Generated {len(records)} FinOps claims | Train split: {n_train} | Held-out Test split: {n_test}")
print("Sample Truthful    :", next(r["text"] for r in records if r["label"] == 0))
print("Sample Hallucinated:", next(r["text"] for r in records if r["label"] == 1))

In [ ]:
# ==============================================================================
# PART 2: MULTI-LAYER RESIDUAL STREAM CAPTURE SWEEP (LAYERS [6,12,18,24,30,35])
# ==============================================================================
import time
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

TARGET_LAYERS = [6, 12, 18, 24, 30, 35]
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# CRITICAL: Left-padding guarantees index [:, -1, :] is ALWAYS the real final prompt token!
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=compute_dtype,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
print(f"Loaded {MODEL_ID} ({model.config.num_hidden_layers} layers, d_model={model.config.hidden_size}).")


def make_activation_capture_hook(cache: dict[int, torch.Tensor], layer_idx: int):
    def _hook(module, inputs, output):
        hidden_states = output[0] if isinstance(output, tuple) else output
        cache[layer_idx] = hidden_states[:, -1, :].detach().to(device="cpu", dtype=torch.float32)
    return _hook


layer_buckets: dict[int, list[torch.Tensor]] = {l: [] for l in TARGET_LAYERS}
batch_cache: dict[int, torch.Tensor] = {}
handles = [
    model.model.layers[l].register_forward_hook(make_activation_capture_hook(batch_cache, l))
    for l in TARGET_LAYERS
]

t_sweep = time.perf_counter()
try:
    with torch.inference_mode():
        for start in range(0, len(records), BATCH_SIZE):
            batch_rows = records[start : start + BATCH_SIZE]
            rendered_prompts = [
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": f"Evaluate this engineering claim:\n{r['text']}"}],
                    add_generation_prompt=True,
                    enable_thinking=False,
                    tokenize=False,
                )
                for r in batch_rows
            ]
            enc = tokenizer(
                rendered_prompts,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            ).to(model.device)
            _ = model(**enc, use_cache=False)
            for l in TARGET_LAYERS:
                layer_buckets[l].append(batch_cache[l])
finally:
    for h in handles:
        h.remove()

torch.cuda.synchronize()
sweep_s = time.perf_counter() - t_sweep
activations_by_layer = {l: torch.cat(layer_buckets[l], dim=0) for l in TARGET_LAYERS}
print(
    f"Captured 1,000 residual stream activations across {len(TARGET_LAYERS)} layers in {sweep_s:.1f}s "
    f"| Tensor shape per layer: {tuple(activations_by_layer[18].shape)}"
)

In [ ]:
# ==============================================================================
# PART 3: LAYER-WISE LINEAR PROBE CLASSIFIER & LAYER 18 CAA STEERING VECTOR
# ==============================================================================
from safetensors.torch import save_file
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate

labels = np.array([r["label"] for r in records], dtype=np.int64)
is_train = np.array([r["split"] == "train" for r in records])
is_test = ~is_train

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
layer_metrics = []

print(f"{'Layer':>5} | {'5-Fold Precision':>16} | {'5-Fold Recall':>13} | {'5-Fold F1':>10} | {'5-Fold AUC':>10}")
print("-" * 68)

for l in TARGET_LAYERS:
    X_all = activations_by_layer[l].numpy()
    clf_cv = LogisticRegression(max_iter=1000, C=0.2, random_state=42)
    scores = cross_validate(
        clf_cv,
        X_all,
        labels,
        cv=cv,
        scoring=["precision", "recall", "f1", "roc_auc"],
    )
    row = {
        "layer": l,
        "cv_precision": float(np.mean(scores["test_precision"])),
        "cv_recall": float(np.mean(scores["test_recall"])),
        "cv_f1": float(np.mean(scores["test_f1"])),
        "cv_roc_auc": float(np.mean(scores["test_roc_auc"])),
    }
    layer_metrics.append(row)
    print(
        f"{l:>5} | {row['cv_precision']:>15.4f} | {row['cv_recall']:>12.4f} | "
        f"{row['cv_f1']:>10.4f} | {row['cv_roc_auc']:>10.4f}"
    )

# Fit production Layer 18 probe on Train Split (800 samples) & evaluate on Held-Out Test Split (200 samples)
X_18 = activations_by_layer[18]
X_train_18, y_train = X_18[is_train].numpy(), labels[is_train]
X_test_18, y_test = X_18[is_test].numpy(), labels[is_test]

probe_18 = LogisticRegression(max_iter=1000, C=0.2, random_state=42)
probe_18.fit(X_train_18, y_train)
test_probs = probe_18.predict_proba(X_test_18)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

heldout_prec = precision_score(y_test, test_preds)
heldout_rec = recall_score(y_test, test_preds)
heldout_f1 = f1_score(y_test, test_preds)
heldout_auc = roc_auc_score(y_test, test_probs)
print(
    f"\nLayer 18 Held-Out Test (n=200): Precision={heldout_prec:.4f} | Recall={heldout_rec:.4f} "
    f"| F1={heldout_f1:.4f} | AUC={heldout_auc:.4f}"
)


def compute_contrastive_steering_vector(truthful_acts: torch.Tensor, hallucinated_acts: torch.Tensor) -> torch.Tensor:
    delta = truthful_acts.mean(dim=0) - hallucinated_acts.mean(dim=0)
    return delta / (delta.norm(p=2) + 1e-12)


truthful_train = X_18[is_train & (labels == 0)]
halluc_train = X_18[is_train & (labels == 1)]
v_truth = compute_contrastive_steering_vector(truthful_train, halluc_train)

probe_w = torch.tensor(probe_18.coef_[0], dtype=torch.float32)
probe_b = torch.tensor([float(probe_18.intercept_[0])], dtype=torch.float32)

artifact_path = OUT_DIR / "layer18_probe_and_steering.safetensors"
save_file(
    {"probe_weight": probe_w, "probe_bias": probe_b, "v_truth": v_truth},
    str(artifact_path),
)
artifact_kb = artifact_path.stat().st_size / 1024.0
print(f"Saved Layer 18 probe + steering vector to {artifact_path} ({artifact_kb:.2f} KB total!)")

In [ ]:
# ==============================================================================
# PART 4: OUTLINES GUIDED JSON + IN-FLIGHT STEERING & <2MS OVERHEAD BENCHMARK
# ==============================================================================
from pydantic import BaseModel, Field


class FinOpsAuditSchema(BaseModel):
    root_cause: str = Field(..., description="Primary GPU memory or latency bottleneck")
    recommended_gpu_vram_gb: int = Field(..., ge=8, le=80)
    projected_savings_pct: int = Field(..., ge=0, le=90)
    grounding_verified: bool


def make_activation_steering_hook(steering_vector: torch.Tensor, alpha: float):
    def _hook(module, inputs, output):
        hidden_states = output[0] if isinstance(output, tuple) else output
        v = steering_vector.to(device=hidden_states.device, dtype=hidden_states.dtype)
        steered = hidden_states + alpha * v
        if isinstance(output, tuple):
            return (steered,) + output[1:]
        return steered
    return _hook


# 1. Benchmark Probe + Steering Hook Latency Overhead on CUDA (<2.0 ms target)
probe_w_gpu = probe_w.to(device=model.device, dtype=torch.float32)
probe_b_gpu = probe_b.to(device=model.device, dtype=torch.float32)

adv_prompt = (
    "A vendor claims we can serve a 70B FP16 dense LLM at 64 concurrent 16k-context streams "
    "inside one 16 GB T4 GPU without quantization. Return a JSON FinOps audit."
)
adv_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": adv_prompt}],
    add_generation_prompt=True,
    enable_thinking=False,
    tokenize=False,
)
adv_enc = tokenizer(adv_text, return_tensors="pt").to(model.device)

bench_cache: dict[int, torch.Tensor] = {}
iters = 40

with torch.inference_mode():
    for _ in range(5):
        _ = model(**adv_enc, use_cache=False)
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(iters):
        _ = model(**adv_enc, use_cache=False)
    torch.cuda.synchronize()
    base_ms = ((time.perf_counter() - t0) / iters) * 1000.0

    def _capture_gpu(module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        bench_cache[18] = hs[:, -1, :].detach().to(dtype=torch.float32)

    h_cap = model.model.layers[18].register_forward_hook(_capture_gpu)
    h_steer = model.model.layers[18].register_forward_hook(
        make_activation_steering_hook(v_truth, alpha=2.5)
    )
    try:
        for _ in range(5):
            _ = model(**adv_enc, use_cache=False)
            _ = torch.sigmoid(torch.dot(bench_cache[18].squeeze(0), probe_w_gpu) + probe_b_gpu)
        torch.cuda.synchronize()

        t1 = time.perf_counter()
        for _ in range(iters):
            _ = model(**adv_enc, use_cache=False)
            p_val = torch.sigmoid(torch.dot(bench_cache[18].squeeze(0), probe_w_gpu) + probe_b_gpu)
        torch.cuda.synchronize()
        hooked_ms = ((time.perf_counter() - t1) / iters) * 1000.0
    finally:
        h_cap.remove()
        h_steer.remove()

hook_overhead_ms = max(0.0, hooked_ms - base_ms)
print(
    f"CUDA Guardrail Latency: Base={base_ms:.2f} ms | Hooked+Probe={hooked_ms:.2f} ms | "
    f"Overhead={hook_overhead_ms:.3f} ms (vs ~550 ms external LLM judge) | Probe P_halluc={p_val.item():.4f}"
)

# 2. Outlines Guided JSON Decoding + Active Layer 18 Steering
outlines_mode = "outlines"
steered_json_raw = ""
h_live = model.model.layers[18].register_forward_hook(
    make_activation_steering_hook(v_truth, alpha=2.5)
)
try:
    try:
        import outlines
        if hasattr(outlines, "from_transformers"):
            o_model = outlines.from_transformers(model, tokenizer)
            steered_json_raw = o_model(adv_text, FinOpsAuditSchema, max_new_tokens=160)
        else:
            o_model = outlines.models.Transformers(model, tokenizer)
            generator = outlines.generate.json(o_model, FinOpsAuditSchema)
            steered_json_raw = generator(adv_text, max_tokens=160)
            if not isinstance(steered_json_raw, str):
                steered_json_raw = steered_json_raw.model_dump_json()
    except Exception as out_err:
        outlines_mode = f"transformers_fallback ({out_err})"
        json_prompt = (
            adv_text
            + '{"root_cause": "'
        )
        j_enc = tokenizer(json_prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out_toks = model.generate(
                **j_enc,
                max_new_tokens=120,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        steered_json_raw = '{"root_cause": "' + tokenizer.decode(
            out_toks[0, j_enc["input_ids"].shape[-1]:], skip_special_tokens=True
        )
finally:
    h_live.remove()

print(f"Guided JSON Mode: {outlines_mode}")
print("Steered JSON Output:\n", steered_json_raw)

In [ ]:
# ==============================================================================
# PART 5: PUSH 1,000-PROMPT ACTIVATION DATASET & PROBE/STEERING REPO TO HF HUB
# ==============================================================================
from datasets import Dataset
from huggingface_hub import HfApi
from safetensors.torch import save_file

# 1. Publish Dataset Repo (1,000 prompts + labels + Layer 18 activation vectors)
acts_18_path = OUT_DIR / "layer18_activations_1000x2560.safetensors"
save_file({"layer18_hidden_states": activations_by_layer[18]}, str(acts_18_path))

ds = Dataset.from_list(records)
ds.push_to_hub(DATASET_REPO_ID)

api = HfApi()
api.upload_file(
    path_or_fileobj=str(acts_18_path),
    path_in_repo="layer18_activations_1000x2560.safetensors",
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    commit_message="feat: add 1,000x2560 Layer 18 residual stream activation tensor",
)
print(f"Dataset pushed to https://huggingface.co/datasets/{DATASET_REPO_ID}")

# 2. Publish Probe + Steering Vector + Benchmark Model Card
probe_telemetry = {
    "model_id": MODEL_ID,
    "gpu_name": gpu_name,
    "num_samples": len(records),
    "target_layers": TARGET_LAYERS,
    "layer_cv_metrics": layer_metrics,
    "layer18_heldout_test": {
        "n_test": int(is_test.sum()),
        "precision": round(heldout_prec, 4),
        "recall": round(heldout_rec, 4),
        "f1": round(heldout_f1, 4),
        "roc_auc": round(heldout_auc, 4),
    },
    "artifact_kb": round(artifact_kb, 2),
    "cuda_base_ms": round(base_ms, 3),
    "cuda_hooked_ms": round(hooked_ms, 3),
    "hook_overhead_ms": round(hook_overhead_ms, 3),
    "outlines_mode": outlines_mode,
}

with open(OUT_DIR / "probe_telemetry.json", "w", encoding="utf-8") as f:
    json.dump(probe_telemetry, f, indent=2)

layer_table_rows = "\n".join(
    f"| Layer {m['layer']} | {m['cv_precision']:.4f} | {m['cv_recall']:.4f} | {m['cv_f1']:.4f} | {m['cv_roc_auc']:.4f} |"
    for m in layer_metrics
)

probe_readme = f"""---
base_model: {MODEL_ID}
datasets:
  - {DATASET_REPO_ID}
tags:
  - activation-steering
  - linear-probes
  - mechanistic-interpretability
  - outlines
  - guardrails
  - finops
  - curons
  - applied-ai-foundry
license: apache-2.0
---

# {PROBE_REPO_ID.split('/')[-1]}: Runtime Residual Stream Linear Probe & CAA Steering Vector

Built as **Prototype 1 (Week 2 / Burst 2)** of the **Applied AI Foundry (`v4.0`)** Sprint (**[`curons.org`](https://curons.org)** | **[`@imhlg`](https://huggingface.co/imhlg)**).

Replaces external LLM-as-a-Judge guardrails (`+550 ms`, `2x` token cost) with an internal PyTorch forward-hook linear probe (`{artifact_kb:.1f} KB` `safetensors`) + Contrastive Activation Addition (CAA) steering vector on Layer 18 of `{MODEL_ID}`.

## 1. Layer-Wise 5-Fold CV Probe Sweep (`n=1,000` FinOps Claims)

| Transformer Layer | 5-Fold Precision | 5-Fold Recall | 5-Fold F1 | 5-Fold ROC-AUC |
| :--- | :--- | :--- | :--- | :--- |
{layer_table_rows}

- **Layer 18 Held-Out Test (`n=200`)**: Precision **`{heldout_prec:.4f}`** | Recall **`{heldout_rec:.4f}`** | F1 **`{heldout_f1:.4f}`** | ROC-AUC **`{heldout_auc:.4f}`**

## 2. Guardrail Latency Benchmark (`{gpu_name}`)

- **Baseline Forward Pass**: `{base_ms:.2f} ms`
- **Capture Hook + Probe + Steering Hook**: `{hooked_ms:.2f} ms`
- **Added Guardrail Overhead**: **`{hook_overhead_ms:.3f} ms`** (vs. `~550 ms` external LLM judge)
"""

with open(OUT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(probe_readme)

api.create_repo(repo_id=PROBE_REPO_ID, repo_type="model", exist_ok=True)
for fname in ("layer18_probe_and_steering.safetensors", "probe_telemetry.json", "README.md"):
    api.upload_file(
        path_or_fileobj=str(OUT_DIR / fname),
        path_in_repo=fname,
        repo_id=PROBE_REPO_ID,
        repo_type="model",
        commit_message="feat: publish Burst 2 Layer 18 linear probe, CAA steering vector & telemetry",
    )

print(f"SUCCESS! Probe & steering repo live at https://huggingface.co/{PROBE_REPO_ID}")
print("ACTION REQUIRED: Click 'Runtime -> Disconnect and delete runtime' NOW to stop Colab CU burn.")